# 03 — AQI Forecasting (t+24h)

**Task**: Cho dữ liệu AQI giờ tại từng thành phố, dự báo AQI **24 giờ tới**.

**Models**:
1. **Baseline**: naive last-value, seasonal naive (same hour last week), rolling mean
2. **XGBoost** — tabular, sử dụng toàn bộ features từ notebook 02
3. **LSTM** — sequence-to-one, input sequence 72h → AQI tại t+24h

**Evaluation**:
- MAE, RMSE, MAPE
- Time-based split: train 01-09/2021, validation 10/2021, test 11-12/2021
- Per-city error breakdown
- **SHAP values** cho XGBoost (feature importance + local explanation)

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.figsize'] = (13, 5)
sns.set_style('whitegrid')

SEED = 42
np.random.seed(SEED)

## 1. Load Features (from notebook 02)

In [ ]:
FEATURES_PATH = Path('./artifacts/features_2021.parquet')
assert FEATURES_PATH.exists(), (
    f'Missing {FEATURES_PATH}. Run notebook 02_feature_engineering.ipynb first.'
)

feats = pd.read_parquet(FEATURES_PATH)
feats['measured_at'] = pd.to_datetime(feats['measured_at'])
print(f'Feature shape: {feats.shape}')
print(f'Date range  : {feats["measured_at"].min()} → {feats["measured_at"].max()}')
print(f'Cities      : {feats["queried_city"].value_counts().to_dict()}')

## 2. Time-Based Train/Val/Test Split

In [ ]:
TRAIN_END = pd.Timestamp('2021-10-01', tz='UTC')
VAL_END   = pd.Timestamp('2021-11-01', tz='UTC')

feats['measured_at'] = pd.to_datetime(feats['measured_at'], utc=True)

train_df = feats[feats['measured_at'] <  TRAIN_END].copy()
val_df   = feats[(feats['measured_at'] >= TRAIN_END) & (feats['measured_at'] < VAL_END)].copy()
test_df  = feats[feats['measured_at'] >= VAL_END].copy()

print(f'Train: {len(train_df):,} rows ({train_df["measured_at"].min()} → {train_df["measured_at"].max()})')
print(f'Val  : {len(val_df):,} rows ({val_df["measured_at"].min()} → {val_df["measured_at"].max()})')
print(f'Test : {len(test_df):,} rows ({test_df["measured_at"].min()} → {test_df["measured_at"].max()})')

drop_cols = ['measured_at', 'queried_city', 'aqi_target']
feature_cols = [c for c in feats.columns if c not in drop_cols]
print(f'\nFeatures ({len(feature_cols)}): {feature_cols[:10]} ...')

## 3. Evaluation Metrics

In [ ]:
def evaluate(y_true, y_pred, name: str = '', verbose: bool = True):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]
    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    eps   = 1e-6
    mape  = float(np.mean(np.abs((y_true - y_pred) / (y_true + eps))) * 100)
    if verbose:
        print(f'{name:<30s} MAE={mae:6.2f}  RMSE={rmse:6.2f}  MAPE={mape:5.1f}%  (n={len(y_true):,})')
    return {'name': name, 'mae': mae, 'rmse': rmse, 'mape': mape, 'n': int(len(y_true))}


results = []

## 4. Baselines

In [ ]:
y_test = test_df['aqi_target'].values

y_naive = test_df['aqi'].values
results.append(evaluate(y_test, y_naive, 'Baseline: naive (aqi_t)'))

y_roll = test_df['aqi_roll_mean_24h'].values
results.append(evaluate(y_test, y_roll, 'Baseline: rolling mean 24h'))

y_roll7 = test_df['aqi_roll_mean_7d'].values
results.append(evaluate(y_test, y_roll7, 'Baseline: rolling mean 7d'))

## 5. Model 1 — XGBoost

In [ ]:
from xgboost import XGBRegressor

cat_city = pd.Categorical(feats['queried_city'])
feats['city_code'] = cat_city.codes

feature_cols_xgb = [c for c in feature_cols if c != 'queried_city']
feature_cols_xgb = feature_cols_xgb + ['city_code']

train_df_x = feats[feats['measured_at'] <  TRAIN_END].copy()
val_df_x   = feats[(feats['measured_at'] >= TRAIN_END) & (feats['measured_at'] < VAL_END)].copy()
test_df_x  = feats[feats['measured_at'] >= VAL_END].copy()

X_tr, y_tr = train_df_x[feature_cols_xgb], train_df_x['aqi_target']
X_va, y_va = val_df_x[feature_cols_xgb],   val_df_x['aqi_target']
X_te, y_te = test_df_x[feature_cols_xgb],  test_df_x['aqi_target']

xgb = XGBRegressor(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method='hist',
    early_stopping_rounds=50,
    random_state=SEED,
    n_jobs=-1,
)

xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=100)

y_pred_xgb = xgb.predict(X_te)
results.append(evaluate(y_te, y_pred_xgb, 'XGBoost'))

print(f'Best iteration: {xgb.best_iteration}')

### 5.1 Feature Importance

In [ ]:
fi = pd.DataFrame({
    'feature': feature_cols_xgb,
    'importance': xgb.feature_importances_,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=fi.head(20), x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title('XGBoost — Top 20 Feature Importances')
plt.tight_layout(); plt.show()
fi.head(20)

### 5.2 SHAP Values — Model Interpretability

In [ ]:
try:
    import shap

    sample_idx = np.random.RandomState(SEED).choice(len(X_te), size=min(500, len(X_te)), replace=False)
    X_shap = X_te.iloc[sample_idx]

    explainer = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_shap)

    shap.summary_plot(shap_values, X_shap, show=False, max_display=15)
    plt.tight_layout(); plt.show()

    shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False, max_display=15)
    plt.tight_layout(); plt.show()
except ImportError:
    print('shap not installed — run: pip install shap')

## 6. Model 2 — LSTM (PyTorch)

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN = 72
HORIZON = 24
torch.manual_seed(SEED)
print(f'Device: {DEVICE}')

In [ ]:
LSTM_FEATURES = ['aqi', 'pm25', 'pm10', 'temperature', 'humidity', 'pressure', 'wind',
                 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_dry_season']

def build_sequences(df_city: pd.DataFrame, feature_list, seq_len: int, horizon: int):
    d = df_city.sort_values('measured_at').reset_index(drop=True)
    X = d[feature_list].astype(float).values
    y = d['aqi'].astype(float).values
    seqs_X, seqs_y, timestamps = [], [], []
    total = len(d) - seq_len - horizon + 1
    for i in range(total):
        x_win = X[i : i + seq_len]
        y_val = y[i + seq_len + horizon - 1]
        if np.isnan(x_win).any() or np.isnan(y_val):
            continue
        seqs_X.append(x_win)
        seqs_y.append(y_val)
        timestamps.append(d['measured_at'].iloc[i + seq_len + horizon - 1])
    return (np.asarray(seqs_X, dtype=np.float32),
            np.asarray(seqs_y, dtype=np.float32),
            pd.to_datetime(timestamps))


X_list, y_list, ts_list, city_list = [], [], [], []
for city in feats['queried_city'].unique():
    sub = feats[feats['queried_city'] == city][['measured_at'] + LSTM_FEATURES].copy()
    if len(sub) < SEQ_LEN + HORIZON + 10:
        continue
    Xc, yc, tc = build_sequences(sub, LSTM_FEATURES, SEQ_LEN, HORIZON)
    X_list.append(Xc); y_list.append(yc); ts_list.append(tc)
    city_list.append(np.repeat(city, len(yc)))

X_all = np.concatenate(X_list, axis=0)
y_all = np.concatenate(y_list, axis=0)
ts_all = pd.Index(np.concatenate([t.values for t in ts_list]))
city_all = np.concatenate(city_list, axis=0)

ts_all = pd.to_datetime(ts_all, utc=True)
tr_mask = ts_all <  TRAIN_END
va_mask = (ts_all >= TRAIN_END) & (ts_all < VAL_END)
te_mask = ts_all >= VAL_END

scaler = StandardScaler()
flat = X_all[tr_mask].reshape(-1, X_all.shape[-1])
scaler.fit(flat)

def apply_scaler(X):
    shape = X.shape
    return scaler.transform(X.reshape(-1, shape[-1])).reshape(shape).astype(np.float32)

X_tr_s = apply_scaler(X_all[tr_mask]); y_tr_s = y_all[tr_mask]
X_va_s = apply_scaler(X_all[va_mask]); y_va_s = y_all[va_mask]
X_te_s = apply_scaler(X_all[te_mask]); y_te_s = y_all[te_mask]

print(f'Sequences — train: {len(X_tr_s):,}, val: {len(X_va_s):,}, test: {len(X_te_s):,}')
print(f'X shape: {X_tr_s.shape}  (batch, seq_len, n_features)')

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class AqiLSTM(nn.Module):
    def __init__(self, n_features, hidden=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0.0,
                            batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


def train_lstm(model, train_loader, val_loader, epochs=20, lr=1e-3, patience=3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.HuberLoss(delta=15.0)
    best_val, best_state, bad = float('inf'), None, 0
    for epoch in range(epochs):
        model.train(); tr_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb); loss = loss_fn(pred, yb)
            loss.backward(); opt.step()
            tr_losses.append(loss.item())
        model.eval(); va_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                va_losses.append(loss_fn(model(xb), yb).item())
        tr, va = float(np.mean(tr_losses)), float(np.mean(va_losses))
        print(f'epoch {epoch+1:02d}  train={tr:.3f}  val={va:.3f}')
        if va < best_val - 1e-3:
            best_val, bad = va, 0
            best_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


train_loader = DataLoader(SeqDataset(X_tr_s, y_tr_s), batch_size=128, shuffle=True)
val_loader   = DataLoader(SeqDataset(X_va_s, y_va_s), batch_size=256)

lstm = AqiLSTM(n_features=X_tr_s.shape[-1], hidden=64, num_layers=2, dropout=0.2).to(DEVICE)
lstm = train_lstm(lstm, train_loader, val_loader, epochs=15, lr=1e-3, patience=3)

In [ ]:
lstm.eval()
preds = []
with torch.no_grad():
    for i in range(0, len(X_te_s), 512):
        xb = torch.from_numpy(X_te_s[i:i+512]).to(DEVICE)
        preds.append(lstm(xb).cpu().numpy())
y_pred_lstm = np.concatenate(preds)
results.append(evaluate(y_te_s, y_pred_lstm, 'LSTM'))

## 7. Model Comparison

In [ ]:
res_df = pd.DataFrame(results).sort_values('mae').reset_index(drop=True)
print(res_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric in zip(axes, ['mae', 'rmse', 'mape']):
    sns.barplot(data=res_df, x='name', y=metric, palette='viridis', ax=ax)
    ax.set_title(metric.upper())
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 8. Per-City Error Breakdown (XGBoost)

In [ ]:
test_df_x = test_df_x.copy()
test_df_x['pred'] = y_pred_xgb
test_df_x['err']  = test_df_x['pred'] - test_df_x['aqi_target']
test_df_x['abs_err'] = test_df_x['err'].abs()

per_city = test_df_x.groupby('queried_city').agg(
    n=('abs_err', 'size'),
    mae=('abs_err', 'mean'),
    rmse=('err', lambda e: float(np.sqrt(np.mean(e**2)))),
    mean_target=('aqi_target', 'mean'),
).round(2).sort_values('mae', ascending=False)
print(per_city)

fig, ax = plt.subplots(figsize=(10, 4))
per_city['mae'].plot.bar(ax=ax, color='steelblue')
ax.set_title('XGBoost MAE per City — Test Set'); ax.set_ylabel('MAE')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

## 9. Predicted vs Actual — Visualization

In [ ]:
cities_to_plot = per_city.index.tolist()[:4]

fig, axes = plt.subplots(len(cities_to_plot), 1, figsize=(14, 3*len(cities_to_plot)), sharex=False)
if len(cities_to_plot) == 1:
    axes = [axes]

for ax, city in zip(axes, cities_to_plot):
    sub = test_df_x[test_df_x['queried_city'] == city].sort_values('measured_at')
    ax.plot(sub['measured_at'], sub['aqi_target'], label='actual',    alpha=0.9)
    ax.plot(sub['measured_at'], sub['pred'],       label='predicted', alpha=0.7)
    ax.set_title(f'{city} — XGBoost 24h Forecast vs Actual')
    ax.set_ylabel('AQI'); ax.legend()
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 10. Save Best Model

In [ ]:
import joblib

OUT_DIR = Path('./artifacts'); OUT_DIR.mkdir(exist_ok=True)
joblib.dump(xgb, OUT_DIR / 'xgb_forecast_model.joblib')

meta = {
    'model_type': 'xgboost',
    'target': 'aqi_target (t+24h)',
    'feature_cols': feature_cols_xgb,
    'city_code_mapping': dict(enumerate(cat_city.categories)),
    'test_metrics': {k: float(v) for k, v in res_df[res_df['name'] == 'XGBoost'].iloc[0].to_dict().items() if k != 'name'},
    'trained_on': f'{train_df_x["measured_at"].min()} → {train_df_x["measured_at"].max()}',
}
import json
with open(OUT_DIR / 'xgb_forecast_meta.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)

print('Saved xgb_forecast_model.joblib + xgb_forecast_meta.json')

## 11. Takeaways cho Production

- **XGBoost thường beat LSTM** trên dataset nhỏ và có structured features — đây là lý do chọn nó cho SageMaker script
- **Lag + rolling features** là đóng góp lớn nhất cho XGBoost — phải chuẩn hoá chính xác trong Glue FE
- **Per-city MAE rất khác nhau**: Ha Noi error lớn hơn do variance cao → có thể cần loss weighting hoặc train per-city
- **SHAP** cho thấy dominant feature là `aqi_roll_mean_24h` + `aqi_lag_1h` — train/serve skew nếu 2 features này bị compute khác nhau
- **Drift monitoring**: cần alarm khi distribution của `aqi_lag_1h` lệch khỏi training set

Production trên SageMaker: xem `sagemaker/forecasting/train.py`